# `07_requests.ipynb`

In [ ]:
# uv add requests
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'

res = requests.get(URL)


raw_data = res.text    # str -> Parsing 안된 데이터
data = res.json()  # dict -> Parsing 된 데이터 (해석됨, 활용 가능)

In [ ]:
# 1인당 1등 당첨 금액 = 'rnk1WnAmt'

data['data']['list'][0]['rnk1WnAmt']  # 1197258718

- 이번주 당첨 정보중 다음 데이터를 추출

```py
...
print(lucky)  # [2, 13, 18, 32, 38, 42]
print(bonus)  # 22
```

In [ ]:
# dict 도 for 로 순회가 가능하다!
d = {'a': 1, 'b': 2, 'c': 3}

for k, v in d.items():
    print(k, v)

In [ ]:
lucky = [1, 2, 3, 4, 5, 6]

my = [1, 2, 3, 4, 5, 6]


# 1: General
count = 0
for ball in lucky:
    if ball in my:
        count += 1

print(count)

# 2: Python 특화
len(set(lucky) & set(my))

In [ ]:
# Main Mission
# 랜덤하게 뽑은 번호 6개와, 실제 당첨번호를 비교하여
# 몇등인지 출력하는 프로그램. 완성하면
# (추가미션) 함수로 잘 만들기 -> 함수로 돌려서 1등 나올때까지 결과 기록

# 1등: 숫자 6개 같음
# 2등: 숫자 5개 같고 + 나머지 하나가 보너스번호
# 3등 ~ 5등: 숫자 5개, 4개, 3개 같음

# 랜덤번호 vs 실제 당첨번호
# -> 우선 고정번호 vs 실제 당첨번호

In [ ]:
import requests

URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
res = requests.get(URL)
data = res.json() 
core_data = data['data']['list'][0]

# 실제 당첨 숫자
lucky = []

for k, v in core_data.items():
    # 아! 모든 로또번호랑 연결된 key에는 'tm' 글자가 들어있군!
    if 'tm' in k:
        # key에 'tm' 들어간 경우에만 해당 value(로또번호)를 추가한다!
        lucky.append(v)
# 보너스 번호
bonus = core_data['bnsWnNo'] 

In [ ]:
import random

# 내가 랜덤하게 뽑은 숫자
my = random.sample(range(1, 46), 6)


match_count = len(set(lucky) & set(my))

if match_count == 6:
    result = '1'
elif match_count == 5 and bonus in my:
    result = '2'
elif match_count == 5:
    result = '3'
elif match_count == 4:
    result = '4'
elif match_count == 3:
    result = '5'
else:
    result = '꽝'

print(result)


In [ ]:
import requests

# 현실 로또 당첨 번호를 API 에서 가져옴
def fetch_lotto_info():
    URL = 'https://www.dhlottery.co.kr/lt645/selectPstLt645Info.do'
    res = requests.get(URL)
    data = res.json() 
    core_data = data['data']['list'][0]

    lucky = []
    for k, v in core_data.items():
        if 'tm' in k:
            lucky.append(v)

    bonus = core_data['bnsWnNo']
    # 최종 return 값은 튜플 (1, 2)
    return lucky, bonus

In [ ]:
# 공주머니 2개랑 보너스를 넣으면 등수를 알려줌
def check_my_luck(my_nums, real_nums, bonus):
    match_count = len(set(my_nums) & set(real_nums))

    if match_count == 6:
        result = '1'
    elif match_count == 5 and bonus in my_nums:
        result = '2'
    elif match_count == 5:
        result = '3'
    elif match_count == 4:
        result = '4'
    elif match_count == 3:
        result = '5'
    else:
        result = '꽝'
    return result

In [ ]:
# 1. 정보 받기
lucky, bonus = fetch_lotto_info()

In [ ]:
import random

dashboard = {
    '1': 0, '2': 0, '3': 0,
    '4': 0, '5': 0, '꽝': 0,
}

# 대시보드에 1등 나온 횟수가 0번이면
while dashboard['1'] == 0:
    # 내번호 랜덤으로 뽑기
    my = random.sample(range(1, 46), 6)
    # 결과 비교하기
    result = check_my_luck(my, lucky, bonus)
    # 대시보드 기록
    dashboard[result] += 1

print(dashboard)


## API Key 관리
1. 터미널에 `uv add python-dotenv` 로 설치
2. 모든 키 파일은 `.env` 파일에 보관 (없으면 생성)
3. 소스코드에서는 `load_dotenv()` 와 `os.getenv()` 를 사용하여 불러옴

In [ ]:
import os
from dotenv import load_dotenv
import requests

# .env 파일 불러오기
load_dotenv()

# 불러온 파일에서 원하는 Key 꺼내기
NAVER_CLIENT_ID = os.getenv('NAVER_CLIENT_ID')
NAVER_CLIENT_SECRET = os.getenv('NAVER_CLIENT_SECRET')

In [ ]:
BASE_URL = 'https://naverapihub.apigw.ntruss.com'
NEWS_URL = '/search/v1/news'

# 인증 관련 헤더
headers = {
    'X-NCP-APIGW-API-KEY-ID': NAVER_CLIENT_ID,
    'X-NCP-APIGW-API-KEY': NAVER_CLIENT_SECRET
}

In [ ]:
URL = BASE_URL + NEWS_URL

# 쿼리 파라미터를 dict 로 작성
params = {
    'query': '엔화',
    'sort': 'sim',
    # 100개 기사를 모아서 (시작은 5개로)
    'display': 100,
}
res = requests.get(URL, headers=headers, params=params)

In [52]:
# 100개 기사를 모아서
# title에 <b> </b> 이상한 태그 없애기 (검색 필요)
# 조건: link URL이 naver 뉴스인 애들만 모아야 함. 100개가 안될 수 있음.
# 간략히 다음과 같은 모양으로 만들기
# news 변수 내용을 csv 로 export 하기
data = res.json()['items']

news = []

for item in data:
    # 2. link에 naver가 없으면 버림
    if 'naver' in item['link']:
        # 1. <b> 없앤걸로 title 교체
        item['title'] = item['title'].replace('<b>', '').replace('</b>', '').replace('&quot;', '')  # 문자열에서 1번 인자를 2번 인자로 교체
        new_item = {
            'title': item['title'],
            'link': item['link']
        }
        news.append(new_item)

for one_news in news:
    requests.get(one_news['link'])

https://n.news.naver.com/mnews/article/003/0014148511?sid=104
https://n.news.naver.com/mnews/article/018/0006358116?sid=101
https://n.news.naver.com/mnews/article/001/0016269394?sid=101
https://n.news.naver.com/mnews/article/015/0005324740?sid=104
https://n.news.naver.com/mnews/article/308/0000038674?sid=101
https://n.news.naver.com/mnews/article/421/0009129662?sid=101
https://n.news.naver.com/mnews/article/016/0002688248?sid=101
https://n.news.naver.com/mnews/article/082/0001394917?sid=101
https://n.news.naver.com/mnews/article/009/0005725075?sid=110
https://n.news.naver.com/mnews/article/421/0009129664?sid=101
https://n.news.naver.com/mnews/article/003/0014145935?sid=104
https://n.news.naver.com/mnews/article/421/0009129696?sid=101
https://n.news.naver.com/mnews/article/015/0005324086?sid=104
https://n.news.naver.com/mnews/article/016/0002688247?sid=101
https://n.news.naver.com/mnews/article/003/0014137178?sid=104
https://n.news.naver.com/mnews/article/001/0016264457?sid=104
https://

In [50]:
import csv

filednames = news[0].keys()

with open('./news.csv', 'w', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=filednames)
    writer.writeheader()
    writer.writerows(news)

## Parsing
1. JSON 문자열 -> dict 로 해석
2. HTML 문자열 -> 구조화 필요 (`BeautifulSoup4`)

In [ ]:
# uv add beautifulsoup4
import requests
from bs4 import BeautifulSoup


def extract_naver_news(url):
    # 네이버 뉴스 아니면 에러 발생
    if 'n.news.naver.com' not in url:
        raise Exception('네이버 뉴스가 아닙니다') 

    res = requests.get(url)

    # res.text 를 해석 완료!
    soup = BeautifulSoup(res.text, 'html.parser')
    
    # 해석한 HTML에서 '#dic_area' 선택자로 추출 -> 글자만 뽑아서 -> 양옆 공백(엔터, 스페이스) 삭제
    news_text = soup.select_one('#dic_area').text.strip()
    return news_text


URL = 'https://n.news.naver.com/article/008/0005405342'
extract_naver_news(URL)


"목요일인 27일 전국 곳곳에 비가 내리겠다. 낮 기온이 최대 34도까지 오르는 등 더위는 계속되겠다. /사진=뉴스1 목요일인 오늘(27일) 전국 곳곳에 비나 소나기가 내리겠다. 낮 기온은 최대 34도까지 오르는 등 무더위는 계속되겠다.기상청에 따르면 이날 아침 최저기온은 21~26도, 낮 최고기온은 29~34도로 예보됐다. 최고 체감온도는 33도 안팎까지 오르겠다.아침부터 낮 사이 경기 북부와 강원에는 5~30㎜의 비가 내리겠다. 충청권과 전북, 대구·경북에는 오전부터 저녁 사이 5~50㎜의 소나기가 내리는 곳이 있겠다.전남권과 경남권, 남해안, 제주도에도 밤까지 비가 이어지겠다. 예상 강수량은 광주·전남과 부산·울산·경남 10~60㎜, 제주도 5~30㎜다. 특히 경남 남해안에는 80㎜ 이상의 많은 비가 내리는 곳도 있겠다.비나 소나기가 내리는 지역에서는 일시적으로 기온이 낮아지겠으나 비가 그친 뒤에는 높은 습도와 함께 다시 무더위가 이어지겠다.주요 도시 예상 최저기온은 △서울 25도 △인천 24도 △춘천 23도 △강릉 24도 △대전 24도 △대구 24도 △전주 25도 △광주 26도 △부산 26도 △여수 26도 △제주 27도 △울릉도·독도 25도 등이다.예상 낮 최고기온은 △서울 31도 △인천 30도 △춘천 31도 △강릉 30도 △대전 33도 △대구 33도 △전주 33도 △광주 33도 △부산 32도 △여수 31도 △제주 34도 △울릉도·독도 29도 등이다.미세먼지 농도는 전 권역에서 '좋음'~'보통' 수준을 보이겠다."

In [9]:
import requests
txt = requests.get('https://n.news.naver.com/article/008/0005405342')
txt.text


'<!doctype html>\n<html lang="ko" data-useragent="python-requests/2.34.2">\n\t<head>\n\t\t<meta charset="utf-8">\n\t\t<meta name="isbot" content="true">\n\t\t<meta http-equiv="X-UA-Compatible" content="IE=edge">\n\t\t<meta name="viewport" content="width=device-width, initial-scale=1.0, maximum-scale=1.0, minimum-scale=1.0, user-scalable=no" />\n\t\t<meta property="og:title" content="[오늘 날씨] &quot;우산 챙겨야&quot;…전국에 비 소식, 낮 최고 34도">\n\t\t<meta property="og:type" content="article">\n\t\t<meta property="og:url" content="https://n.news.naver.com/article/008/0005405342">\n\t\t<meta property="og:image" content="https://imgnews.pstatic.net/image/008/2026/08/27/0005405342_001_20260827060113418.jpg?type&#x3D;w800">\n\t\t<meta property="og:description" content="목요일인 오늘(27일) 전국 곳곳에 비나 소나기가 내리겠다. 낮 기온은 최대 34도까지 오르는 등 무더위는 계속되겠다. 기상청에 따르면 이날 아침 최저기온은 21~26도, 낮 최고기온은 29~34도로 예보됐다. 최고">\n\t\t<meta property="og:article:author" content="머니투데이 | 네이버">\n\t\t<meta name="twitter:card" content="summary_large_

### OpenAI 사용하기
- Web API 방식 -> HTTP 방식으로 OpenAI 서버에 요청을 보내서 응답을 받음
- SDK 방식 -> API를 더 쓰기 쉽게 만들어 준 개발자 친화적 키트(Kit) 

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
#uv add openai
from openai import OpenAI

#OpenAI 기능을 사용할 클라이언트 생성
client = OpenAI(
  api_key=os.getenv('OPENAI_API_KEY')
)

system_msg = '너는 매우 착한 댓글을 만들어주는 AI야. 기사 내용을 보고 긍정적인 댓글을 만들어줘'
user_msg = ''

# client.responses.create(
#   model = 'gpt-4.1-mini',
#   # system message
#   instructions = system_msg,
#   # user message
#   input = user_msg

# )

gpt_res = client.responses.create(
  model = 'gpt-4.1-mini',
  # system message
  instructions = system_msg,
  # user message
  input = user_msg

)
#출력결과 텍스트로
gpt_res.output_text